In [1]:
!pip install -q ultralytics

In [2]:
import numpy as np, torch, pandas as pd, random
from pathlib import Path
from PIL import Image
from torchvision.ops import batched_nms
from ultralytics import YOLO

INPUT = Path('/kaggle/input')
W17 = next(INPUT.rglob('run17_best_finetuned.pt'))
RDD = next(p for p in INPUT.rglob('RoadScan_yolo') if (p/'test_US/images').is_dir())
IMG, LBL = RDD/'test_US/images', RDD/'test_US/labels'
EXT = {'.jpg', '.jpeg', '.png'}
net = YOLO(str(W17))

paths_all = sorted(p for p in IMG.glob('*') if p.suffix.lower() in EXT)
sz = pd.Series([max(Image.open(p).size) for p in paths_all])
print('US images:', len(paths_all), '| الجهة الطويلة min/median/max:', sz.min(), sz.median(), sz.max(), flush=True)

def iou1(a, bs):
    lo = np.maximum(a[:2], bs[:, :2]); hi = np.minimum(a[2:], bs[:, 2:])
    inter = np.maximum(hi - lo, 0).prod(1)
    return inter / np.maximum((a[2:]-a[:2]).prod() + (bs[:, 2:]-bs[:, :2]).prod(1) - inter, 1e-9)

def load_gt(lp, w, h):
    g = []
    if lp.exists():
        for line in lp.read_text().splitlines():
            if line.strip():
                c, x, y, bw, bh = map(float, line.split())
                g.append([c, (x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h])
    return np.array(g).reshape(-1, 5)

def plain(sz, conf):
    def f(net, im):
        r = net.predict(im, imgsz=sz, conf=conf, iou=0.7, max_det=300, verbose=False)[0]
        return r.boxes.data.cpu().numpy().reshape(-1, 6)
    return f

def tiled(S, conf, tile=640, overlap=0.2, add_full=False):
    def f(net, im):
        w, h = im.size; sc = min(1.0, S / max(w, h))
        im2 = im.resize((round(w*sc), round(h*sc)), Image.BILINEAR) if sc < 1 else im
        W_, H_ = im2.size; step = int(tile * (1 - overlap))
        def starts(L):
            if L <= tile: return [0]
            s = list(range(0, L - tile + 1, step))
            if s[-1] != L - tile: s.append(L - tile)
            return s
        crops, offs = [], []
        for y in starts(H_):
            for x in starts(W_):
                crops.append(im2.crop((x, y, min(x+tile, W_), min(y+tile, H_)))); offs.append((x, y))
        out = []
        for i in range(0, len(crops), 16):
            rs = net.predict(crops[i:i+16], imgsz=tile, conf=conf, iou=0.7, max_det=300, verbose=False)
            for r, (x, y) in zip(rs, offs[i:i+16]):
                b = r.boxes.data.cpu().numpy().reshape(-1, 6).copy()
                b[:, [0, 2]] += x; b[:, [1, 3]] += y; out.append(b)
        b = np.concatenate(out) if out else np.zeros((0, 6), dtype=np.float32)
        b[:, :4] /= sc
        if add_full: b = np.concatenate([b, plain(1280, conf)(net, im)])
        if len(b):
            keep = batched_nms(torch.tensor(b[:, :4]), torch.tensor(b[:, 4]), torch.tensor(b[:, 5]).long(), 0.5).numpy()
            b = b[keep]
        return b
    return f

def evaluate(net, paths, lbl_dir, predict):
    tp = fp = fn = 0
    for ip in paths:
        im = Image.open(ip).convert('RGB'); w, h = im.size
        g = load_gt(Path(lbl_dir)/(ip.stem + '.txt'), w, h)
        d = predict(net, im); d = d[np.argsort(-d[:, 4])] if len(d) else d
        used = set()
        for pp in d:
            ids = [i for i in range(len(g)) if i not in used and int(g[i, 0]) == int(pp[5])]
            hit = False
            if ids:
                ov = iou1(pp[:4], g[ids, 1:]); k = int(ov.argmax())
                if ov[k] >= 0.5: used.add(ids[k]); hit = True
            tp += hit; fp += (not hit)
        fn += len(g) - len(used)
    return dict(P=tp/max(tp+fp, 1), R=tp/max(tp+fn, 1), F1=2*tp/max(2*tp+fp+fn, 1), TP=tp, FP=fp, FN=fn)

random.seed(0)
paths = random.sample(paths_all, min(300, len(paths_all)))
methods = {'plain640': plain(640, .46), 'plain1280': plain(1280, .46),
           'tile1600': tiled(1600, .46), 'tile2000': tiled(2000, .46),
           'tile2000+full': tiled(2000, .46, add_full=True)}
rows = []
for name, fn in methods.items():
    r = evaluate(net, paths, LBL, fn); r['method'] = name; rows.append(r); print(r, flush=True)
res = pd.DataFrame(rows); res.to_csv('/kaggle/working/tiling_us.csv', index=False); display(res)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


StopIteration: 

In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd
ps = [p for p in Path('/kaggle/input').rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}]
s = pd.Series([max(Image.open(p).size) for p in ps])
print(len(s), s.min(), s.median(), s.max())